# W1.2 bulk OCR on Colab: Bodhan IndicOCR over the scanned corpus

Madhya Kanoon Saathi, DEC-001 / DEC-015 / DEC-053. Runtime: **GPU (T4)**. 21 documents, 635 pages.

**Before running:**
1. Unzip `data/ocr/bulk/mks_ocr_bulk.zip` and upload the folder `mks_ocr_bulk/` (`jobs.json` + `pdfs/`) to your
   Google Drive root.
2. Colab secret `HF_TOKEN` (key icon, notebook access ON): the same Hugging Face **read** token used for the bake-off.
3. Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom.
4. **Resumable:** every page is skipped if its output exists. After a timeout, reconnect and run all cells again.

Order: indexed documents first (134 pages), gated ones after; the 256-page MPLRC Hindi consolidation is last.
Outputs: `mks_ocr_bulk/out/<doc>/pNNNN.md` + `.json`, `out/run_info.json`, and at the end `mks_ocr_bulk_out.zip`.
Download that ZIP and unzip it to `data/ocr/bulk/` in the repo. Nothing here prints secrets.

## 0. GPU check (stops early on a CPU runtime)

In [ ]:
import torch

# Bodhan needs a GPU. A CPU runtime ships the CPU-only torch build and fails later with
# "Torch not compiled with CUDA enabled". Fix: Runtime -> Change runtime type -> T4 GPU.
assert torch.cuda.is_available(), (
    "No GPU: switch to Runtime -> Change runtime type -> T4 GPU (if T4 is greyed out, "
    "the free GPU quota is used up; try again later)."
)
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0))

In [ ]:
import datetime
import hashlib
import json
import pathlib
import time

from google.colab import drive

drive.mount("/content/drive")
ROOT = pathlib.Path("/content/drive/MyDrive/mks_ocr_bulk")
JOBS = json.loads((ROOT / "jobs.json").read_text(encoding="utf-8"))
OUT = ROOT / "out"
OUT.mkdir(exist_ok=True)
INFO_PATH = OUT / "run_info.json"
INFO = json.loads(INFO_PATH.read_text()) if INFO_PATH.exists() else {"pages": {}}


def save_info():
    INFO_PATH.write_text(json.dumps(INFO, indent=1, ensure_ascii=False))


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


bad = [j["doc_id"] for j in JOBS["jobs"] if sha256(ROOT / j["file"]) != j["sha256"]]
assert not bad, f"sha256 mismatch (re-upload these PDFs): {bad}"
print(len(JOBS["jobs"]), "documents,", JOBS["total_pages"], "pages; all sha256 match")

## 1. Renderer: same pypdfium2 version as the repo lockfile, 300 dpi

In [ ]:
!pip -q install "{JOBS['renderer']}"
import pypdfium2 as pdfium

INFO["renderer"] = JOBS["renderer"]
INFO["dpi"] = JOBS["dpi"]
save_info()

## 2. Bodhan IndicOCR, pinned to the bake-off revision
Usage follows the model card at https://huggingface.co/bodhan-ai/indic-ocr (checked 2026-11-10).

In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download

# same revision as the W0.5 bake-off run (data/ocr/out/run_info.json)
REVISION = "cd50d301d0e17e8ecb32fc49c8ccbd7914dcfc25"
tok = userdata.get("HF_TOKEN")
REPO = snapshot_download("bodhan-ai/indic-ocr", revision=REVISION, token=tok)
del tok
INFO["bodhan"] = {
    "repo": "bodhan-ai/indic-ocr",
    "revision": REVISION,
    "licence": "Indic Open Model License v1.0",
    "table_format": "markdown",
}
save_info()
print("revision", REVISION)

In [ ]:
!cd "$REPO" && ./install.sh

## 3. OCR every page (skips pages already done)

In [ ]:
import sys

sys.path.insert(0, REPO)
from indic_ocr import IndicOCR

parser = IndicOCR.from_pretrained(REPO, table_format="markdown")
TMP = pathlib.Path("/content/page.png")
started = time.time()
for j in JOBS["jobs"]:
    d = OUT / j["doc_id"].replace("@", "_")
    d.mkdir(exist_ok=True)
    pdf = None
    for n in range(1, j["pages"] + 1):
        key = f"{j['doc_id']}#p{n}"
        if (d / f"p{n:04d}.md").exists():
            continue
        if pdf is None:
            pdf = pdfium.PdfDocument(str(ROOT / j["file"]))
        img = pdf[n - 1].render(scale=JOBS["dpi"] / 72).to_pil().convert("RGB")
        img.save(TMP)
        t0 = time.time()
        page = parser.parse(str(TMP))
        js = json.dumps(page, ensure_ascii=False, indent=1)
        (d / f"p{n:04d}.json").write_text(js, encoding="utf-8")
        # the .md is written last, so its presence means the page is done
        (d / f"p{n:04d}.md").write_text(page["markdown"], encoding="utf-8")
        INFO["pages"][key] = {"seconds": round(time.time() - t0, 1), "size": list(img.size)}
        save_info()
        print(key, f"{INFO['pages'][key]['seconds']}s", len(page["markdown"]), "chars")
    if pdf is not None:
        pdf.close()
print("session done in", round((time.time() - started) / 60, 1), "min")

## 4. Check and package

In [ ]:
import shutil

missing = [
    f"{j['doc_id']}#p{n}"
    for j in JOBS["jobs"]
    for n in range(1, j["pages"] + 1)
    if not (OUT / j["doc_id"].replace("@", "_") / f"p{n:04d}.md").exists()
]
print("pages missing:", len(missing), missing[:10])
if not missing:
    INFO["finished_at"] = datetime.datetime.now(datetime.UTC).isoformat()
    save_info()
    shutil.copy(ROOT / "jobs.json", OUT / "jobs.json")
    print(shutil.make_archive(str(ROOT.parent / "mks_ocr_bulk_out"), "zip", ROOT, "out"))